In [17]:
prev_val

'-'

In [13]:
import os
import pandas as pd

# 1. 상세메뉴 목록 불러오기
menu_file_path = r"C:\ksmindpks\ai_x\source\Pks_Develop\식당대12중53소132상세메뉴380분류.csv"
df_menu = pd.read_csv(menu_file_path)
split_menus = df_menu["상세메뉴"].dropna().apply(lambda x: [m.strip() for m in x.split(",")])
menu_list = [item for sublist in split_menus for item in sublist]
# 순서 유지한 상세메뉴 리스트
seen = set()
ordered_menu_list = [x for x in menu_list if not (x in seen or seen.add(x))]

# 2. 열 이름 매핑 함수 (포함되면 대체, 단 '구이'와 '전복버터구이'는 그대로 유지)
def match_column(col_name, menu_list, already_mapped):
    if col_name in ['구이', '전복버터구이']:
        return col_name
    for menu in menu_list:
        if col_name in menu and menu not in already_mapped:
            return menu
    return col_name

# 3. 병합 대상 경로
folder_path = r"C:\ksmindpks\ai_x\source\Pks_Develop\레서피"
excel_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".xlsx")])

# 4. 기준 시트명 확보
first_file_path = os.path.join(folder_path, excel_files[0])
first_xls = pd.ExcelFile(first_file_path)
reference_sheet_names = first_xls.sheet_names[:8]

# 5. 병합 결과 저장 구조
merged_dataframes = {}
mapping_logs = {}

# 6. 병합 수행
for idx, sheet_name in enumerate(reference_sheet_names):
    all_data_dict = {}
    col_map_log = {}
    already_mapped_menus = set()
    
    for file in excel_files:
        file_path = os.path.join(folder_path, file)
        xls = pd.ExcelFile(file_path)
        
        if idx >= len(xls.sheet_names):
            continue
        
        df = xls.parse(xls.sheet_names[idx], dtype=str).fillna("-")
        df.columns.values[0] = "항목"
        df = df.set_index("항목")
        
        # 열 이름 매핑
        new_columns = []
        for col in df.columns:
            mapped_col = match_column(col, ordered_menu_list, already_mapped_menus)
            if mapped_col != col:
                col_map_log[col] = mapped_col
                already_mapped_menus.add(mapped_col)
            new_columns.append(mapped_col)
        df.columns = new_columns
        
        # 병합 수행
        for item, row in df.iterrows():
            item = item.strip()
            if item not in all_data_dict:
                all_data_dict[item] = {}
            for col, val in row.items():
                if val != "-":
                    prev_val = all_data_dict[item].get(col, "-")
                    if prev_val == "-":
                        all_data_dict[item][col] = val
                    elif val != prev_val and val not in prev_val.split(" / "):
                        all_data_dict[item][col] = prev_val + " / " + val
    
    # 380개 메뉴 포함 여부 확인 및 추가
    if "항목" not in all_data_dict:
        all_data_dict["항목"] = {}
    for menu in ordered_menu_list:
        if menu not in all_data_dict["항목"].values():
            all_data_dict["항목"][menu] = "-"
    
    # DataFrame 구성
    merged_df = pd.DataFrame.from_dict(all_data_dict, orient="index")
    merged_df.index.name = "항목"
    merged_df = merged_df.reset_index().copy()
    merged_df = merged_df.fillna("-")
    
    # 열 순서 유지 (항목은 맨 앞, 380개 메뉴 순서대로)
    cols = ["항목"] + ordered_menu_list
    merged_df = merged_df[cols]
    
    merged_dataframes[sheet_name] = merged_df
    mapping_logs[sheet_name] = col_map_log

# 7. 결과 저장
output_file = os.path.join(folder_path, "병합_레시피_포함매핑적용_380개메뉴포함.xlsx")
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in merged_dataframes.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# 8. 매핑 로그 출력
print("[열 이름 포함 기반 매핑 결과]")
for sheet, mappings in mapping_logs.items():
    if mappings:
        print(f"\n시트: {sheet}")
        for original, mapped in mappings.items():
            print(f"- '{original}' → '{mapped}' (포함기반 매핑)")

[열 이름 포함 기반 매핑 결과]

시트: 주요식재료
- '불고기' → '불고기도시락' (포함기반 매핑)
- '타코' → '타코야키' (포함기반 매핑)
- '스크램블' → '스크램블에그' (포함기반 매핑)
- '허니머스타드' → '허니머스타드치킨' (포함기반 매핑)
- '뿌링클' → '뿌링클치킨' (포함기반 매핑)
- '치즈가루' → '치즈가루치킨' (포함기반 매핑)
- '어니언' → '어니언치킨' (포함기반 매핑)
- '윙' → '치킨윙' (포함기반 매핑)
- '다리' → '치킨다리' (포함기반 매핑)
- '가슴살' → '치킨가슴살' (포함기반 매핑)
- '콤비네이션' → '콤비네이션피자' (포함기반 매핑)
- '딥디쉬' → '딥디쉬피자' (포함기반 매핑)
- '씬크러스트' → '씬크러스트피자' (포함기반 매핑)
- '크리스피' → '크리스피피자' (포함기반 매핑)
- '롱블랙' → '롱블랙커피' (포함기반 매핑)

시트: 양념소스
- '불고기' → '불고기도시락' (포함기반 매핑)
- '타코' → '타코야키' (포함기반 매핑)
- '스크램블' → '스크램블에그' (포함기반 매핑)
- '허니머스타드' → '허니머스타드치킨' (포함기반 매핑)
- '뿌링클' → '뿌링클치킨' (포함기반 매핑)
- '치즈가루' → '치즈가루치킨' (포함기반 매핑)
- '어니언' → '어니언치킨' (포함기반 매핑)
- '윙' → '치킨윙' (포함기반 매핑)
- '다리' → '치킨다리' (포함기반 매핑)
- '가슴살' → '치킨가슴살' (포함기반 매핑)
- '콤비네이션' → '콤비네이션피자' (포함기반 매핑)
- '딥디쉬' → '딥디쉬피자' (포함기반 매핑)
- '씬크러스트' → '씬크러스트피자' (포함기반 매핑)
- '크리스피' → '크리스피피자' (포함기반 매핑)
- '롱블랙' → '롱블랙커피' (포함기반 매핑)

시트: 육수베이스
- '불고기' → '불고기도시락' (포함기반 매핑)
- '타코' → '타코야키' (포함기반 매핑)
- '스크램블' → '스크램블에그' (포함기반 매핑